## Budget on an L4

18 runs: 6 rungs x 3 seeds. The full-run settings are tuned for an **L4 with
High-RAM**, not an A100.

| | per seed | per rung (3 seeds) |
|---|---|---|
| `L0_flat`, `L1_bottleneck`, `L2_graph` | ~15 min | ~45 min |
| `L3_weak`, `L4_contrastive`, `L5_full` | ~22 min | ~65 min |

**Roughly 5-6 hours in total.** Each epoch prints its own duration and an ETA,
so you will know within two minutes whether that estimate holds for your runtime.

Split it across sessions with `RUNGS` in cell 1, for example
`RUNGS = ['L0_flat', 'L1_bottleneck', 'L2_graph']` in one session and the rest in
the next. Everything resumes, and because output goes straight to Drive a
disconnect costs at most the epoch in progress.

Settings that keep it inside an L4: batch 32, `max_length` 192, 6 epochs,
6,000 weak rows, bfloat16 autocast, and no redundant final pass over the
training set. To cut further, lower `weak_max_rows` or drop to
`epochs=5` - but change it once, before you start, so every rung stays
comparable.

## 1. Mount Drive and configure

All paths point at Drive directly. Nothing is written to local disk.

In [ ]:
import os, sys, json, math, warnings
import pandas as pd
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
PROCESSED = f'{DRIVE_BASE}/processed'          # notebook 02
GRAPH     = f'{DRIVE_BASE}/graph'              # notebook 03
WEAK      = f'{DRIVE_BASE}/weak_labels/weak_labels.parquet'   # notebook 04
BASELINES_DIR = f'{DRIVE_BASE}/baselines'      # notebook 05

# True -> tiny model and subset, about 3 minutes, separate folder
QUICK_TEST = True
SUFFIX = '_quicktest' if QUICK_TEST else ''
RESULTS = f'{DRIVE_BASE}/proposed{SUFFIX}'

# Which rungs to train in this session. None means all six.
RUNGS = None

os.makedirs(RESULTS, exist_ok=True)
print('mode    :', 'QUICK TEST' if QUICK_TEST else 'FULL RUN')
print('writing :', RESULTS)

missing = [p for p in (PROCESSED, GRAPH) if not os.path.isdir(p)]
assert not missing, f'missing prerequisites: {missing} - run notebooks 02 and 03'
print('processed:', sorted(os.listdir(PROCESSED)))
print('graph    :', sorted(os.listdir(GRAPH)))
print('weak     :', 'found' if os.path.exists(WEAK) else 'MISSING (run notebook 04)')
print('baselines:', 'found' if os.path.isdir(BASELINES_DIR) else 'not yet (notebook 05)')

## 2. Get the code

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q $REPO $CHECKOUT

need = ['src/symptograph/proposed.py', 'src/symptograph/metrics.py']
missing = [f for f in need if not os.path.isfile(f'{CHECKOUT}/{f}')]
assert not missing, f'stale checkout, missing {missing} - run: !rm -rf {CHECKOUT}'
sys.path.insert(0, f'{CHECKOUT}/src')
!git -C $CHECKOUT log --oneline -1

### GPU and batch size

Batch size is chosen from the available memory, since a Pro runtime may give you
anything from a T4 to an A100 and a larger batch is simply faster.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    # roberta-base at 192 tokens is small; an L4's 24 GB holds batch 32 easily
    BATCH = 64 if total_gb > 40 else 32 if total_gb > 20 else 16 if total_gb > 14 else 8
    bf16 = torch.cuda.is_bf16_supported()
    print(f'{gpu}  {total_gb:.0f} GB  ->  batch {BATCH}, '
          f'autocast {"bfloat16" if bf16 else "float16"}')
    if 'L4' in gpu:
        print('L4 detected: the full-run settings below are tuned for it.')
else:
    BATCH = 8
    print('No GPU. Runtime -> Change runtime type -> L4 GPU + High-RAM.')

### Hugging Face access for MentalRoBERTa

`mental/mental-roberta-base` is a **gated** repo. The gate is auto-approve, not
a review queue, so this takes about a minute:

1. Open <https://huggingface.co/mental/mental-roberta-base> while signed in and
   click **Agree and access repository**.
2. Create a **read** token at <https://huggingface.co/settings/tokens>.
3. Either add it to Colab secrets as `HF_TOKEN` (key icon in the left sidebar,
   with notebook access enabled), or paste it when the cell below asks.

The cell then **verifies access before training starts**, so a permissions
problem costs five seconds instead of surfacing hours into a run.

If you would rather not deal with the gate, set `ENCODER` below to an ungated
option. `roberta-base` is a perfectly respectable choice - just rename the run
honestly, because it is not a domain-pretrained model and the paper should not
claim otherwise.

In [ ]:
from symptograph.proposed import check_encoder_access, ENCODER_OPTIONS

# The encoder every rung and the mentalroberta baseline will use.
ENCODER = 'mental/mental-roberta-base'

print('options:')
for k, v in ENCODER_OPTIONS.items():
    print(f'  {k:34s} {v}')

status = check_encoder_access(ENCODER)
if not status['ok']:
    # try to authenticate, then re-check
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        print('\nusing HF_TOKEN from Colab secrets')
    except Exception:
        pass
    if not token:
        from getpass import getpass
        token = getpass('\nHugging Face read token (blank to skip): ').strip() or None
    if token:
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
        status = check_encoder_access(ENCODER)

print()
if status['ok']:
    print(f'ACCESS OK: {ENCODER}')
else:
    print(f'NO ACCESS to {ENCODER}')
    print(f'  {status["reason"]}')
    print('\nEither finish the two steps above, or set ENCODER to an ungated')
    print('option and re-run this cell. Do not start the full run until this')
    print('says ACCESS OK, or you will fail on the first rung.')

## 3. The shared training protocol

One config object drives all 18 runs. Only the five component switches change
between rungs; every other hyperparameter is held fixed, and each rung's
`config.json` records exactly what it used.

Encoder and head use different learning rates (2e-5 and 1e-3) because the encoder
is pretrained and the heads are not. The `mentalroberta` baseline used the same
encoder rate, so `L0_flat` stays comparable to it.

In [ ]:
from symptograph.proposed import (ProposedConfig, run_ladder, LADDER,
                                  LADDER_ORDER, FULL_MODEL)
from symptograph.metrics import compare_runs, headline_table, set_publication_style
from symptograph.jsonio import read_json, repair_json_tree
from symptograph.preprocess import Ontology

onto = Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')
set_publication_style(font_size=20)

if QUICK_TEST:
    # BERT-mini: small (11M) but genuinely PRETRAINED, so the quick test can
    # actually detect a model that is not learning. An untrained encoder scores
    # at chance no matter what, which once hid a real bug in the bottleneck.
    # ~10 minutes on an L4, ~0.8 GB of checkpoints.
    cfg = ProposedConfig(epochs=3, seeds=(0, 1, 2), batch_size=32,
                         eval_batch_size=64, max_length=128,
                         limit_train=1500, limit_eval=600,
                         graph_dim=64,
                         encoder_name='google/bert_uncased_L-4_H-256_A-4',
                         strict_encoder=True,
                         save_optimizer_state=False,
                         weak_label_path=WEAK, weak_max_rows=600)
else:
    # Tuned for an L4 (24 GB). max_length 192 still covers about 80% of posts
    # (median 73 words, p90 188), and 6,000 weak rows roughly doubles the
    # symptom supervision without tripling the epoch time.
    cfg = ProposedConfig(epochs=6, seeds=(0, 1, 2), batch_size=BATCH,
                         eval_batch_size=BATCH * 2, max_length=192,
                         encoder_name=ENCODER,
                         weak_label_path=WEAK,
                         weak_max_rows=6_000,
                         amp_dtype='auto',
                         strict_encoder=True,   # never silently substitute RoBERTa
                         checkpoint_every=1)

print('epochs          ', cfg.epochs)
print('seeds           ', cfg.seeds)
print('batch           ', cfg.batch_size, '| max_length', cfg.max_length)
print('lr encoder/head ', cfg.lr, '/', cfg.head_lr)
print('task weights    ', cfg.task_weights)
print('weak rows cap   ', cfg.weak_max_rows)
print('selection       ', cfg.selection_metric, '(validation)')
print('autocast        ', cfg.amp_dtype)
print('encoder         ', 'tiny-random (untrained)' if cfg.tiny_random_encoder
      else f'{cfg.encoder_name}  (strict={cfg.strict_encoder})')
print()
for name, flags in LADDER:
    on = [k.replace('use_', '') for k, v in flags.items() if v]
    print(f'  {name:16s} {", ".join(on) if on else "(none)"}')

## 4. Train the ladder

Each epoch prints **train and validation** side by side, so divergence is visible
immediately rather than at the end.

**If the session drops, re-run this cell.** Finished epochs are skipped and
`history.csv` is appended to, never rewritten. To split the work across sessions,
set `RUNGS` in cell 1, for example `RUNGS = ['L0_flat', 'L1_bottleneck']`.

In [ ]:
results = run_ladder(cfg, PROCESSED, GRAPH, RESULTS,
                     rungs=RUNGS, onto=onto, resume=True, verbose=True)
print('\ntrained:', list(results))

### Repair any JSON damaged by the Drive mount

Google Drive's FUSE mount does not reliably truncate a file reopened in `"w"`
mode, so rewriting a slightly shorter file on resume could leave the tail of the
old one and produce `JSONDecodeError: Extra data`. Writes are atomic now, but
this repairs anything damaged by an earlier session - it recovers the valid
leading object, so no retraining is needed.

In [ ]:
_ = repair_json_tree(RESULTS)

### Which encoder actually ran

Worth one cell, because a failed download falls back silently unless
`strict_encoder=True`, and the difference between MentalRoBERTa and RoBERTa is
exactly what the paper claims.

In [ ]:
# Confirm every rung used the encoder you asked for. A silent fallback would
# mean publishing RoBERTa numbers under the MentalRoBERTa name.
rows = []
for rung in LADDER_ORDER:
    cfg_path = f'{RESULTS}/{rung}/seed_0/config.json'
    if not os.path.isfile(cfg_path):
        continue
    c = read_json(cfg_path)
    rows.append({'rung': rung, 'encoder_used': c.get('encoder'),
                 'requested': c.get('encoder_requested'),
                 'fell_back': c.get('encoder_fallback_used'),
                 'params_M': round(c.get('trainable_parameters', 0) / 1e6, 2)})
display(pd.DataFrame(rows).set_index('rung'))

fell = [r['rung'] for r in rows if r['fell_back']]
if fell:
    print(f'WARNING: these rungs fell back to another encoder: {fell}')
    print('Their results are NOT for the requested model. Delete and re-run them.')
elif rows:
    print('All rungs used:', rows[0]['encoder_used'])

### Sanity check: is the model actually learning?

The single most useful check in this notebook. A model can run cleanly end to
end and still be broken - an earlier version shared one scoring vector across
all 22 symptoms, so every bottleneck rung sat at F1 = 0.000 while the pipeline
reported success.

On the quick test (BERT-mini, 1,500 rows, 3 epochs) expect roughly:

| | pass | broken |
|---|---|---|
| val symptom AUROC | **> 0.55** | ~0.50 |
| val symptom F1 | **> 0.20** | 0.000 |
| val severity QWK | **> 0.00** | 0.000 everywhere |

The full run should comfortably exceed these.

In [ ]:
checks = []
for rung in LADDER_ORDER:
    h_path = f'{RESULTS}/{rung}/seed_0/history.csv'
    if not os.path.isfile(h_path):
        continue
    h = pd.read_csv(h_path)
    last = h.iloc[-1]
    auroc = last.get('val_group_all_macro_f1')
    checks.append({
        'rung': rung,
        'val_symF1': round(last.get('val_group_all_macro_f1', float('nan')), 4),
        'val_symAUROC': round(last.get('val_group_all_macro_auroc', float('nan')), 4),
        'val_sevQWK': round(last.get('val_severity_depression_qwk', float('nan')), 4),
        'max_loss': round(max(h.train_loss.max(), h.val_loss.max()), 3),
    })
cdf = pd.DataFrame(checks).set_index('rung')
display(cdf)

problems = []
if (cdf['max_loss'] > 100).any():
    problems.append('loss exploded (>100) - numerical instability')
if (cdf['val_symF1'] <= 0.001).all():
    problems.append('symptom F1 is 0 everywhere - the model predicts no positives')
if (cdf['val_symAUROC'] < 0.52).all():
    problems.append('symptom AUROC at chance - the encoder is not learning')
if (cdf['val_sevQWK'].abs() < 1e-9).all():
    problems.append('severity QWK is 0 everywhere - the severity head is dead')

if problems:
    print('FAILED sanity checks:')
    for p in problems:
        print('  -', p)
    print('\nDo not start the full run. Investigate first.')
else:
    print('Sanity checks passed: the model is learning on all rungs.')

## 5. Epoch-wise history

`history.csv` carries about 310 columns: every per-symptom metric, every group
aggregate and every task metric, for **both** train and validation.

In [ ]:
TRAINED = [r for r in LADDER_ORDER
           if os.path.isfile(f'{RESULTS}/{r}/seed_0/history.csv')]

for rung in TRAINED:
    h = pd.read_csv(f'{RESULTS}/{rung}/seed_0/history.csv')
    print(f'\n=== {rung} (seed 0): {len(h)} epochs, {len(h.columns)} columns ===')
    cols = ['epoch', 'train_loss', 'val_loss',
            'train_group_all_macro_f1', 'val_group_all_macro_f1',
            'train_severity_depression_qwk', 'val_severity_depression_qwk',
            'train_composite', 'val_composite']
    display(h[[c for c in cols if c in h.columns]].round(4))

## 6. Publication figures for the full model

Every figure is 300 dpi with font size 20, ready to drop into the paper.

In [ ]:
from IPython.display import Image, display as show

plot_dir = f'{RESULTS}/{FULL_MODEL}/seed_0/plots'
if os.path.isdir(plot_dir):
    for p in sorted(os.listdir(plot_dir)):
        print(p)
        show(Image(filename=f'{plot_dir}/{p}', width=820))

## 7. Mean and standard deviation per rung

In [ ]:
for rung in TRAINED:
    meta = f'{RESULTS}/{rung}/aggregate/aggregate_metrics.json'
    if not os.path.isfile(meta):
        continue
    agg = read_json(meta)
    on = [k.replace('use_', '') for k, v in agg['components'].items() if v]
    print(f'\n=== {rung} === components: {", ".join(on) or "(none)"}'
          f' | best epoch per seed: {agg["per_seed_best_epoch"]}')
    display(headline_table(agg['aggregate'], label=rung)[['metric', 'formatted']]
            .set_index('metric'))

## 8. Comparison A - against the ablations

The ladder plot is the central ablation figure: each point adds one novelty, so a
rising line is the evidence that each component earns its place.

In [ ]:
ablation_runs = {r: f'{RESULTS}/{r}' for r in TRAINED}
combined_ab, tables_ab = compare_runs(
    ablation_runs, f'{RESULTS}/comparison_vs_ablations',
    title='SymptoGraph-LLM ablation ladder',
    ladder_order=LADDER_ORDER, font_size=20)

wide = combined_ab.pivot(index='metric', columns='model', values='formatted')
display(wide[[c for c in LADDER_ORDER if c in wide.columns]])

In [ ]:
# per-rung delta: what each novelty actually bought
rows = []
prev = None
for rung in TRAINED:
    t = tables_ab[rung].set_index('metric')
    cur = t['mean'].get('composite')
    rows.append({'rung': rung, 'composite': None if cur is None else round(cur, 4),
                 'delta_vs_previous': None if (cur is None or prev is None)
                                      else round(cur - prev, 4)})
    prev = cur if cur is not None else prev
deltas = pd.DataFrame(rows).set_index('rung')
display(deltas)
deltas.to_csv(f'{RESULTS}/comparison_vs_ablations/ladder_deltas.csv')

for p in sorted(os.listdir(f'{RESULTS}/comparison_vs_ablations')):
    if p.endswith('.png'):
        print(p)
        show(Image(filename=f'{RESULTS}/comparison_vs_ablations/{p}', width=880))

## 9. Comparison B - against the baselines

Kept deliberately separate from the ablation comparison: one answers *does the
architecture beat the prior art*, the other answers *does each component
contribute*. Mixing them into one table muddles both questions.

In [ ]:
from symptograph.baselines import BASELINES

baseline_runs = {b: f'{BASELINES_DIR}/{b}' for b in BASELINES
                 if os.path.isfile(f'{BASELINES_DIR}/{b}/aggregate/aggregate_metrics.json')}
if not baseline_runs:
    print(f'No baseline results in {BASELINES_DIR}. Run notebook 05 first '
          f'(with QUICK_TEST matching this notebook).')
else:
    runs = dict(baseline_runs)
    runs['SymptoGraph-LLM'] = f'{RESULTS}/{FULL_MODEL}'
    combined_bl, tables_bl = compare_runs(
        runs, f'{RESULTS}/comparison_vs_baselines',
        title='SymptoGraph-LLM vs baselines', font_size=20)
    display(combined_bl.pivot(index='metric', columns='model', values='formatted'))
    show(Image(filename=f'{RESULTS}/comparison_vs_baselines/comparison.png', width=900))

### Held-out test results

From each seed's **best-validation** checkpoint, so the test split never
influenced selection.

In [ ]:
test_rows = []
for label, path in {**{b: f'{BASELINES_DIR}/{b}' for b in baseline_runs},
                    'SymptoGraph-LLM': f'{RESULTS}/{FULL_MODEL}'}.items():
    meta = f'{path}/aggregate/aggregate_metrics.json'
    if os.path.isfile(meta):
        by = read_json(meta).get('aggregate_by_split', {})
        if 'test' in by:
            test_rows.append(headline_table(by['test'], label=label))
if test_rows:
    tc = pd.concat(test_rows, ignore_index=True)
    display(tc.pivot(index='metric', columns='model', values='formatted'))
    tc.to_csv(f'{RESULTS}/comparison_vs_baselines/test_comparison.csv', index=False)

## 10. Selective prediction (N6)

Only `L5_full` has the uncertainty head. **AURC is lower-is-better**: it is the
area under the risk-coverage curve, so a falling curve means the model's
uncertainty is informative and deferring the least confident cases genuinely
reduces error - which is the operational claim behind N6.

In [ ]:
rows = []
for seed in cfg.seeds:
    p = f'{RESULTS}/{FULL_MODEL}/seed_{seed}/performance_metrics.json'
    if not os.path.isfile(p):
        continue
    sel = read_json(p).get('selective', {}).get('test')
    if sel:
        rows.append({'seed': seed, 'aurc': round(sel['aurc'], 4),
                     'risk@100%': round(sel['risk_at_full_coverage'], 4),
                     'risk@80%': sel['risk_at_80_coverage'],
                     'risk@50%': sel['risk_at_50_coverage']})
if rows:
    sdf = pd.DataFrame(rows).set_index('seed')
    display(sdf)
    print('mean +/- std:')
    display(pd.DataFrame({'mean': sdf.mean().round(4),
                          'std': sdf.std(ddof=1).round(4)}))
    show(Image(filename=f'{RESULTS}/{FULL_MODEL}/seed_0/plots/risk_coverage.png',
               width=700))
else:
    print('No selective results yet - train L5_full.')

## 11. What to report

Everything is already on Drive. This cell just lists the artefacts.

In [ ]:
print('results root:', RESULTS)
!du -sh "$RESULTS"/* 2>/dev/null | sort -h
print()
lb = f'{RESULTS}/comparison_vs_ablations/leaderboard.md'
if os.path.isfile(lb):
    print(open(lb).read()[:1800])

## Notes

**Where each novelty lives in the code**, in `src/symptograph/proposed.py`:

- **N1** symptom-wise attention queries initialised from the sentence embeddings
  of the clinical definitions; severity, risk and stress are predicted *only*
  from the concept vector, so nothing bypasses the bottleneck.
- **N2** dense adjacency `A = A0 + dA`, where `A0` is notebook 03's graph and `dA`
  is learned under an L1 penalty. Inspect `model.delta` after training to report
  which edges the data added.
- **N3** weak labels enter as extra training rows whose per-symptom loss weight is
  the calibrated confidence from notebook 04.
- **N5** supervised contrastive loss, with positives defined by symptom-profile
  Jaccard overlap rather than a coarse class label.
- **N6** a per-symptom log-variance head with learned loss attenuation; that same
  variance is the abstention score in section 10.

**Reading the ladder honestly.** With 3 seeds, a gain smaller than the standard
deviation is not evidence. Suicide risk QWK in particular has only 340 training
and 74 test rows and swings widely. If a rung's delta is within noise, report it
as such and say so - a clean negative result on one component is far more
credible than six components that all conveniently help.

**A fairness point for N3.** Rungs L3 to L5 see weak labels that the baselines
never did. For the headline claim against prior art, also run notebook 05 with
`use_weak_labels=True` and compare against that, otherwise part of the gain is
extra data rather than architecture.

**Still to do for the full paper:** N4 (the symptom-grounded rationale generator
and its faithfulness evaluation), the concept-intervention experiment, and the
external evaluation on the held-out IMHI subsets.

### Troubleshooting

`Refusing to resume: ... came from a different architecture` means a checkpoint
in that folder was built with different settings, usually a quick test. The
message names each changed field; delete that run folder and start it again.
Quick-test output is kept in `proposed_quicktest/` for exactly this reason.

Checkpoints are full fp32 including optimiser state, about 1.5 GB each, written
to Drive every epoch. If that I/O dominates your epoch time, set
`checkpoint_every=2`.

**If you hit CUDA out of memory on the L4**, halve the batch size in cell 3
(`BATCH = 16`) and restart the runtime. Do it before the first rung, so all 18
runs share one batch size - changing it midway makes the rungs incomparable.

**`Could not load the requested encoder 'mental/mental-roberta-base'`** means the
gated repo was refused. That is `strict_encoder=True` protecting you: without it
the run would have quietly used `roberta-base` and still reported
"MentalRoBERTa". Accept the licence and authenticate in the access cell above,
or pick an ungated `ENCODER`.

**Check your baselines too.** `baselines.py` had the same hole. Confirm what
notebook 05 actually ran:

```python
import json
c = json.load(open(f'{DRIVE_BASE}/baselines/mentalroberta/seed_0/config.json'))
print(c.get('encoder'), '| fell back:', c.get('encoder_fallback_used'))
```

If it says `roberta-base`, that baseline is RoBERTa and needs re-running with
access granted, or renaming.

**Watching your credits.** `RUNGS` is the control: run two or three rungs per
session and stop. Nothing is lost between sessions because every artefact,
including the optimiser state, is already on Drive.

## Disk use, and clearing the quick test

Checkpoint size follows the encoder, so plan for it:

| Run | Per seed | All 18 runs |
|---|---|---|
| quick test (`tiny_random_encoder`, no optimiser state) | ~27 MB | **~0.5 GB** |
| full run (MentalRoBERTa, fp32 + optimiser) | ~2 GB | **~36 GB** |

If an earlier quick test used a full-size encoder it will have written roughly
34 GB. Delete it before the full run:

```python
!du -sh "$DRIVE_BASE/proposed_quicktest"
!rm -rf "$DRIVE_BASE/proposed_quicktest"
```

Only checkpoints are large. If you want to keep the metrics but reclaim the
space after a run finishes, the `.pt` files are the only thing worth deleting -
`history.csv`, `performance_metrics.json` and the plots are a few MB in total:

```python
# keeps every metric and figure, drops resume capability
!find "$RESULTS" -name 'checkpoint_last.pt' -delete
```